# Класифікація рентгенівських знімків: NORMAL / PNEUMONIA

У цьому ноутбуці використовується MobileNetV2 для навчальної класифікації рентгенівських знімків грудної клітки.

In [ ]:
import kagglehub

# Завантаження датасету Chest X-Ray Images (Pneumonia) з Kaggle.
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")

print("Path to dataset files:", path)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf

from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

print("TensorFlow version:", tf.__version__)

In [ ]:
# Шляхи до каталогів датасету.
dataset_path = os.path.join(path, "chest_xray")
train_path = os.path.join(dataset_path, "train")
test_path = os.path.join(dataset_path, "test")

# Перевіряємо, чи каталоги існують.
for folder in (train_path, test_path):
    if not os.path.isdir(folder):
        raise FileNotFoundError(f"Не знайдено каталог датасету: {folder}")

train_set_classes = sorted(
    name for name in os.listdir(train_path)
    if os.path.isdir(os.path.join(train_path, name))
)

print("Classes in train set:", train_set_classes)

In [ ]:
# Перевіряємо кількість зображень у кожному класі.
for category in train_set_classes:
    folder = os.path.join(train_path, category)
    print(f"{category}: {len(os.listdir(folder))} images")

In [ ]:
# Візуалізуємо по одному прикладу з кожного класу.
plt.figure(figsize=(12, 5))

for i, category in enumerate(train_set_classes):
    folder = os.path.join(train_path, category)
    image_files = [
        name for name in os.listdir(folder)
        if os.path.isfile(os.path.join(folder, name))
    ]

    if not image_files:
        print(f"У каталозі {category} немає зображень.")
        continue

    image = tf.keras.utils.load_img(
        os.path.join(folder, sorted(image_files)[0]),
        target_size=(224, 224)
    )

    plt.subplot(1, len(train_set_classes), i + 1)
    plt.imshow(image, cmap="gray")
    plt.title(category)
    plt.axis("off")

plt.tight_layout()
plt.show()

## Підготовка даних та аугментація

Для навчальної вибірки застосовуємо помірну аугментацію. Для валідаційної та тестової вибірок залишаємо тільки preprocessing, щоб оцінювання не залежало від випадкових перетворень.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

# Аугментація застосовується тільки до навчальних зображень.
train_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
    validation_split=0.2,
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True
)

# Окремий генератор без аугментації для валідації.
validation_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input,
    validation_split=0.2
)

train_generator = train_datagen.flow_from_directory(
    train_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="training",
    shuffle=True,
    seed=SEED
)

val_generator = validation_datagen.flow_from_directory(
    train_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    subset="validation",
    shuffle=False,
    seed=SEED
)

test_datagen = ImageDataGenerator(
    preprocessing_function=tf.keras.applications.mobilenet_v2.preprocess_input
)

test_generator = test_datagen.flow_from_directory(
    test_path,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode="binary",
    shuffle=False
)

print("Class mapping:", train_generator.class_indices)

# Перевіряємо, що всі набори використовують однакове зіставлення класів.
if train_generator.class_indices != val_generator.class_indices:
    raise ValueError("Зіставлення класів у train та validation не збігається.")

if train_generator.class_indices != test_generator.class_indices:
    raise ValueError("Зіставлення класів у train та test не збігається.")

## Створення моделі MobileNetV2

Використовуємо MobileNetV2 як попередньо навчену базову мережу. Якщо завантажити ImageNet-ваги неможливо, ноутбук автоматично перейде до ініціалізації без попередньо навчених ваг.

In [ ]:
try:
    base_model = tf.keras.applications.MobileNetV2(
        weights="imagenet",
        include_top=False,
        input_shape=(224, 224, 3)
    )
    print("MobileNetV2 завантажено з ImageNet-вагами.")
except Exception as error:
    print("Не вдалося завантажити ImageNet-ваги:", error)
    print("Використовуємо MobileNetV2 без попередньо навчених ваг.")
    base_model = tf.keras.applications.MobileNetV2(
        weights=None,
        include_top=False,
        input_shape=(224, 224, 3)
    )

# Заморожуємо базову мережу на першому етапі навчання.
base_model.trainable = False

# Додаємо власну класифікаційну голову для двох класів.
inputs = layers.Input(shape=(224, 224, 3))
x = base_model(inputs, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.2)(x)
outputs = layers.Dense(1, activation="sigmoid")(x)

model = models.Model(inputs, outputs)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

## Навчання моделі

EarlyStopping зупинить навчання, якщо якість на валідаційній вибірці перестане покращуватися. ReduceLROnPlateau зменшить швидкість навчання, якщо валідаційна втрата перестане знижуватися.

In [ ]:
EPOCHS = 10

callbacks = [
    EarlyStopping(
        monitor="val_loss",
        patience=3,
        restore_best_weights=True,
        verbose=1
    ),
    ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.2,
        patience=2,
        min_lr=1e-6,
        verbose=1
    )
]

history = model.fit(
    train_generator,
    validation_data=val_generator,
    epochs=EPOCHS,
    callbacks=callbacks
)

In [ ]:
# Зберігаємо навчену модель, щоб не довелося навчати її повторно.
model.save("pneumonia_mobilenetv2.keras")
print("Модель збережено у файлі pneumonia_mobilenetv2.keras")

## Прогнозування на тестовій вибірці

Після навчання перевіряємо модель на тестових зображеннях, які не використовувалися для оновлення ваг. Для кожного зображення модель повертає ймовірність класу з індексом 1. Поріг 0.5 використовується для перетворення ймовірності на клас.

In [ ]:
# Визначаємо назви класів та їхні індекси.
class_indices = test_generator.class_indices
class_names = [name for name, index in sorted(class_indices.items(), key=lambda item: item[1])]

print("Class mapping:", class_indices)
print("Class names by index:", class_names)

# Скидаємо генератор, щоб прогнозування починалося з першого зображення.
test_generator.reset()

# Отримуємо ймовірності прогнозів для тестового набору.
y_prob = model.predict(test_generator, verbose=1).ravel()

# Фактичні мітки та прогнозовані мітки.
y_true = test_generator.classes
y_pred = (y_prob >= 0.5).astype(int)

print("Number of test images:", len(y_true))
print("Number of predictions:", len(y_pred))

## Перегляд окремих прогнозів

Порівнюємо прогноз моделі з фактичною міткою для кількох тестових зображень. Це допомагає побачити не лише загальні метрики, а й окремі приклади правильних та помилкових класифікацій.

In [ ]:
import pandas as pd

# Індекс 1 відповідає ймовірності, яку повертає sigmoid-вихід моделі.
predicted_names = [class_names[index] for index in y_pred]
actual_names = [class_names[index] for index in y_true]

predictions_df = pd.DataFrame({
    "Image": test_generator.filenames,
    "Actual class": actual_names,
    "Predicted class": predicted_names,
    "Probability of class 1": np.round(y_prob, 4),
    "Correct prediction": y_true == y_pred
})

# Виводимо перші 15 результатів.
display(predictions_df.head(15))

## Метрики якості моделі

Обчислюємо Accuracy, Precision, Recall, F1-score та Specificity. Для медичного навчального прикладу особливо важливо стежити за Recall класу PNEUMONIA: ця метрика показує, яку частку зображень із пневмонією модель змогла виявити.

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Знаходимо індекси класів за їхніми назвами.
if "PNEUMONIA" not in class_indices or "NORMAL" not in class_indices:
    raise ValueError(f"Очікували класи NORMAL та PNEUMONIA, отримали: {class_indices}")

normal_index = class_indices["NORMAL"]
pneumonia_index = class_indices["PNEUMONIA"]

# У цій бінарній моделі sigmoid повертає ймовірність класу з індексом 1.
# За потреби перетворюємо її на ймовірність саме класу PNEUMONIA.
pneumonia_probability = y_prob if pneumonia_index == 1 else 1 - y_prob

# Матриця помилок у порядку NORMAL, PNEUMONIA.
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[normal_index, pneumonia_index]
)

tn, fp, fn, tp = cm.ravel()

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, pos_label=pneumonia_index, zero_division=0)
recall = recall_score(y_true, y_pred, pos_label=pneumonia_index, zero_division=0)
f1 = f1_score(y_true, y_pred, pos_label=pneumonia_index, zero_division=0)
specificity = tn / (tn + fp) if (tn + fp) > 0 else 0.0

try:
    roc_auc = roc_auc_score((y_true == pneumonia_index).astype(int), pneumonia_probability)
except ValueError:
    roc_auc = float("nan")

print("Classification report:")
print(classification_report(
    y_true,
    y_pred,
    labels=[normal_index, pneumonia_index],
    target_names=["NORMAL", "PNEUMONIA"],
    zero_division=0
))

print("Additional metrics:")
print(f"Accuracy:    {accuracy:.4f}")
print(f"Precision:   {precision:.4f}")
print(f"Recall:      {recall:.4f}")
print(f"F1-score:    {f1:.4f}")
print(f"Specificity: {specificity:.4f}")
print(f"ROC-AUC:     {roc_auc:.4f}")

## Матриця помилок (Confusion Matrix)

Матриця показує кількість правильних і помилкових прогнозів для кожного класу. Рядки відповідають фактичним класам, стовпці — прогнозам моделі.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["NORMAL", "PNEUMONIA"]
)
display.plot(cmap="Blues", values_format="d", ax=ax, colorbar=False)
plt.title("Confusion Matrix — Pneumonia Classification")
plt.tight_layout()
plt.show()

print(f"True negatives (NORMAL predicted as NORMAL): {tn}")
print(f"False positives (NORMAL predicted as PNEUMONIA): {fp}")
print(f"False negatives (PNEUMONIA predicted as NORMAL): {fn}")
print(f"True positives (PNEUMONIA predicted as PNEUMONIA): {tp}")

## Підсумкова оцінка моделі

Наведений нижче автоматичний коментар допомагає інтерпретувати отримані тестові метрики. Він не замінює професійну медичну оцінку та не означає, що модель придатна для клінічного використання.

In [ ]:
print("MODEL EVALUATION")
print("=" * 50)

if recall >= 0.90:
    print("Recall: модель виявила більшість тестових зображень із пневмонією.")
elif recall >= 0.70:
    print("Recall: результат помірний; частина зображень із пневмонією залишається невиявленою.")
else:
    print("Recall: результат низький; модель пропускає значну частину зображень із пневмонією.")

if precision >= 0.90:
    print("Precision: більшість позитивних прогнозів моделі правильні.")
elif precision >= 0.70:
    print("Precision: є помітна частка хибнопозитивних прогнозів.")
else:
    print("Precision: багато позитивних прогнозів можуть бути помилковими.")

if f1 >= 0.90:
    print("F1-score: баланс Precision та Recall високий.")
elif f1 >= 0.70:
    print("F1-score: баланс Precision та Recall помірний.")
else:
    print("F1-score: баланс Precision та Recall потребує покращення.")

print("\nFinal test metrics:")
print(f"Accuracy    = {accuracy:.4f}")
print(f"Precision   = {precision:.4f}")
print(f"Recall      = {recall:.4f}")
print(f"F1-score    = {f1:.4f}")
print(f"Specificity = {specificity:.4f}")
print(f"ROC-AUC     = {roc_auc:.4f}")

print("\nConclusion: these results describe performance on this dataset's test split only.")
print("The model is an educational experiment, not a medical diagnostic system.")